In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# DN = 'C://work/dev/python/progs/texts/sec_bert/'
DN = '/home/jovyan/work/sec_bert/'

import os
os.chdir(DN)

In [ ]:
from sklearn.metrics import (average_precision_score, log_loss, confusion_matrix,
                            precision_recall_fscore_support, f1_score)

import matplotlib.pyplot as plt


import os

from ruamel.yaml import YAML
import pandas as pd
import numpy as np
import joblib

from collections import defaultdict
from itertools import chain

import click
import json

import torch
import torch.nn as nn
from torch.optim.lr_scheduler import ExponentialLR, MultiStepLR
from torch.utils.data import DataLoader, Dataset


from transformers import BertTokenizer, BertForSequenceClassification
from transformers import AutoTokenizer, AutoModelForMaskedLM, BertConfig, AutoModel
from transformers import DataCollatorWithPadding
from transformers import RobertaTokenizer, RobertaModel

DEVICE = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

import sys
sys.path.append('.')
from src.funcs import set_seed
from src.funcs import metric_multi
from src.funcs import get_opt_thresh, get_preds
from src.funcs import get_conf_df
from src.spec_nn_funcs import TextDFDataset, TextModelClass
from ruamel.yaml import YAML

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB

from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.preprocessing import RobustScaler

from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import RobustScaler, StandardScaler

from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.tree import DecisionTreeClassifier

In [ ]:


conf = YAML().load(open('params.yaml'))
conf_ttp = YAML().load(open('dvc_pipes/ttp/params_ttp.yaml'))
conf_bert = YAML().load(open('dvc_pipes/bert/params_bert.yaml'))
conf_bert_ttp = YAML().load(open('dvc_pipes/bert_ttp/params_bert_ttp.yaml'))

set_seed(conf['seed'])

In [ ]:
bert_type = conf_bert['nn_bert']['bert_type']


In [ ]:
VALID_BATCH_SIZE = conf_bert['nn']['batch_size']
TRAIN_BATCH_SIZE = conf_bert['nn']['batch_size']
MAX_SEQ_LENGTH = conf_bert['nn']['maxlen']

checkpoint = 'data/external/models/SecureBERT_Plus/snapshots/4c48ccdb8d2019f179b07dfa27656c655394d78e'
tokenizer = RobertaTokenizer.from_pretrained(checkpoint)
tokenizer_opts = {'max_length':MAX_SEQ_LENGTH, 'return_tensors':"pt", 'padding':True, 'truncation':True, 'add_special_tokens':True}


# Предсказания техник

In [ ]:
mlb_ttp = joblib.load(conf['prep_text']['ttp_mlb_fn'])


In [ ]:
data_ttp = pd.read_csv(conf_ttp['feat_gen_ttp']['data_fn'])
data_ttp['target'] = data_ttp['target'].map(lambda x: eval(x))
data_ttp['ttp'] = data_ttp['ttp'].map(lambda x: eval(x))



In [ ]:
model_bert_ttp = torch.load(f'{conf_bert_ttp["nn_bert_ttp"]["model_fn"]}')



In [ ]:
feat_ttp = pd.read_csv(conf_ttp['feat_eng_ttp']['feat_final_fn'])



In [ ]:
tr_ttp_ds = TextDFDataset(data_ttp.query('split=="tr"').reset_index(drop=True), tokenizer=tokenizer, tokenizer_opts=tokenizer_opts)
val_ttp_ds = TextDFDataset(data_ttp.query('split=="val"').reset_index(drop=True), tokenizer=tokenizer, tokenizer_opts=tokenizer_opts)
ts_ttp_ds = TextDFDataset(data_ttp.query('split=="ts"').reset_index(drop=True), tokenizer=tokenizer, tokenizer_opts=tokenizer_opts)

tr_ttp_ld = DataLoader(tr_ttp_ds, batch_size = TRAIN_BATCH_SIZE, shuffle = False, collate_fn = DataCollatorWithPadding(tokenizer=tokenizer))
val_ttp_ld = DataLoader(val_ttp_ds, batch_size = VALID_BATCH_SIZE, shuffle = False, collate_fn = DataCollatorWithPadding(tokenizer=tokenizer))
ts_ttp_ld = DataLoader(ts_ttp_ds, batch_size = VALID_BATCH_SIZE, shuffle = False, collate_fn = DataCollatorWithPadding(tokenizer=tokenizer))


In [ ]:
Y_ttp_val_proba = np.array(get_preds(model_bert_ttp, ld=val_ttp_ld)['pred'])
Y_ttp_tr_proba = np.array(get_preds(model_bert_ttp, ld=tr_ttp_ld)['pred'])
Y_ttp_ts_proba = np.array(get_preds(model_bert_ttp, ld=ts_ttp_ld)['pred'])

In [ ]:
thresh_ttp_l = get_opt_thresh(y_true = np.array(data_ttp.loc[data_ttp.split=='tr', 'target'].values.tolist()), 
                          probas = Y_ttp_tr_proba, mlb = mlb_ttp, 
                          opt_metric=conf['train_eval_model']['opt_metric'], 
                          thresh_space_l=np.arange(0.001, 1, 0.002), dump_fn = None)


In [ ]:
ttp_df = pd.concat([data_ttp[['sentence', 'ttp', 'labels',	'url', 'target', 'split']].query('split=="tr"').assign(proba_ttp = Y_ttp_tr_proba.tolist()),
          data_ttp[['sentence', 'ttp', 'labels',	'url', 'target', 'split']].query('split=="val"').assign(proba_ttp = Y_ttp_val_proba.tolist()),
           data_ttp[['sentence', 'ttp', 'labels',	'url', 'target', 'split']].query('split=="ts"').assign(proba_ttp = Y_ttp_ts_proba.tolist())
          ], axis=0, ignore_index=True)

ttp_df['pred_ttp'] = ttp_df['proba_ttp'].map(lambda x: [int(val>=thresh) for val, thresh in zip(x, thresh_ttp_l)])

ttp_df['pred_str_ttp'] = ttp_df['pred_ttp'].map(lambda x: mlb_ttp.inverse_transform(np.array([x]))[0])


In [ ]:
ttp_df.query('split in ["val", "ts"]').head(2)

In [ ]:
_, res_l = metric_multi(np.array(ttp_df.query('split in ["val", "ts"]')['target'].values.tolist()), np.array(ttp_df.query('split in ["val", "ts"]')['pred_ttp'].values.tolist()), f1_score)
    
bad_df = pd.DataFrame({'qual':res_l, 'class':mlb_ttp.classes_}).sort_values(by='qual')#.head(20)

bad_df.to_csv(f'data/temp/bad_cls/cls_{conf['seed']}.csv', index=False)

In [ ]:
bad_cls_l = bad_df['class'].tolist()
res_df = ttp_df[(ttp_df['split'].isin(["val", "ts"]))&(ttp_df['ttp'].map(lambda x: any([it in x for it in bad_cls_l])))]

res_df[['sentence', 'ttp', 'labels', 'pred_str_ttp']].sort_values(by='ttp').to_csv(f'data/temp/bad_cls/sentence_{conf['seed']}.csv', index=False)

# После дампа всех значений на разных сидах

- соединяем классы плохие
- ищем по общей метрике самые плохие 15 и для таких классов

In [ ]:
DN = 'data/temp/bad_cls/'
bad_cls_df = pd.concat([pd.read_csv(f'{DN}/{it}').assign(seed=os.path.splitext(it)[0].split('cls_')[1]) for it in os.listdir(DN) if 'cls_' in it])

In [ ]:
bad_cls_df.groupby('class')['qual'].mean().sort_values().reset_index().head(10)


In [ ]:
bad_sent_df = pd.concat([pd.read_csv(f'{DN}/{it}').assign(seed=os.path.splitext(it)[0].split('sentence_')[1]) for it in os.listdir(DN) if 'sentence_' in it])

In [ ]:
bad_sent_df['ttp'] = bad_sent_df['ttp'].map(lambda x: eval(x))

In [ ]:
bad_sent_df.explode('ttp').merge(bad_cls_df.groupby('class')['qual'].mean().sort_values().reset_index().head(10), 
                                 left_on='ttp', right_on='class', how='right')\
                            .sort_values(by=['qual'])#.to_csv('data/temp/bad_cls/bad10CLS.csv', sep=';', index=False)

In [ ]:
cls_l = bad_cls_df.groupby('class')['qual'].mean().sort_values().head(20).index.tolist()

bad_sent_df[bad_sent_df.ttp.map(lambda x: any([it in x for it in cls_l]))].sort_values(by='ttp')